In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\halil melih\4BitirmeProject\bitirme_project")
KEY_PATH = PROJECT_DIR / "secrets" / "gcp-speech.json"

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(KEY_PATH)
print("Credential set:", os.environ["GOOGLE_APPLICATION_CREDENTIALS"])


Credential set: C:\Users\halil melih\480proje\470_project\bitirme_project\secrets\gcp-speech.json


In [79]:
from pathlib import Path

# Proje kök klasörü (ipynb'yi bitirme_project içinde çalıştırdığını varsayar)
PROJECT_DIR = Path(".").resolve()

# İSTENEN VERİ YOLU (tercih edilen): data/raw/911_recordings/audio
AUDIO_DIR = PROJECT_DIR / "data" / "raw" / "911_recordings" / "audio"


print("PROJECT_DIR:", PROJECT_DIR)
print("AUDIO_DIR:", AUDIO_DIR)
print("AUDIO_DIR exists?:", AUDIO_DIR.exists())

mp3_files = sorted(AUDIO_DIR.glob("*.mp3"))
print("MP3 count:", len(mp3_files))
mp3_files[:5]


PROJECT_DIR: C:\Users\halil melih\480proje\470_project\bitirme_project
AUDIO_DIR: C:\Users\halil melih\480proje\470_project\bitirme_project\data\raw\911_recordings\audio
AUDIO_DIR exists?: True
MP3 count: 707


[WindowsPath('C:/Users/halil melih/480proje/470_project/bitirme_project/data/raw/911_recordings/audio/call_1.mp3'),
 WindowsPath('C:/Users/halil melih/480proje/470_project/bitirme_project/data/raw/911_recordings/audio/call_10.mp3'),
 WindowsPath('C:/Users/halil melih/480proje/470_project/bitirme_project/data/raw/911_recordings/audio/call_100.mp3'),
 WindowsPath('C:/Users/halil melih/480proje/470_project/bitirme_project/data/raw/911_recordings/audio/call_101.mp3'),
 WindowsPath('C:/Users/halil melih/480proje/470_project/bitirme_project/data/raw/911_recordings/audio/call_102.mp3')]

In [80]:
import os
from google.cloud import speech

creds = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
print("GOOGLE_APPLICATION_CREDENTIALS:", creds)


GOOGLE_APPLICATION_CREDENTIALS: C:\Users\halil melih\480proje\470_project\bitirme_project\secrets\gcp-speech.json


In [81]:
import os
import subprocess
from dataclasses import dataclass

@dataclass
class TranscriptionResult:
    transcript: str
    confidence: float | None
    language_code: str
    flac_path: str | None = None

def ensure_flac_mono_16k(input_path: str, out_dir: str = ".tmp_audio") -> str:
    # MP3/WAV gibi dosyaları Google STT için güvenli format olan:
    # mono + 16kHz + FLAC'a dönüştürür (ffmpeg gerekir).
    os.makedirs(out_dir, exist_ok=True)
    base = os.path.splitext(os.path.basename(input_path))[0]
    out_path = os.path.join(out_dir, f"{base}.flac")

    cmd = [
        "ffmpeg", "-y", "-i", input_path,
        "-ac", "1",          # mono
        "-ar", "16000",      # 16kHz
        "-vn",
        out_path
    ]
    # ffmpeg çıktısını sessize al
    subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return out_path

import os, subprocess, tempfile
from pathlib import Path

def split_to_flac_chunks(input_path: str, chunk_seconds: int = 45) -> list[str]:
    """
    MP3/WAV -> 16k mono FLAC chunk'lara böler.
    chunk_seconds'ı 30-60 aralığında tutarsan 10MB limitine takılmaz.
    """
    out_dir = Path(tempfile.mkdtemp(prefix="stt_chunks_"))
    out_pattern = str(out_dir / "chunk_%03d.flac")

    cmd = [
        "ffmpeg", "-y", "-i", input_path,
        "-ac", "1", "-ar", "16000",
        "-af", "highpass=f=200, lowpass=f=3000, dynaudnorm",
        "-f", "segment", "-segment_time", str(chunk_seconds),
        out_pattern
    ]
    subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

    chunks = sorted(str(p) for p in out_dir.glob("chunk_*.flac"))
    return chunks

def transcribe_local_file_chunked(input_path: str, language_code: str = "en-US", chunk_seconds: int = 45):
    chunks = split_to_flac_chunks(input_path, chunk_seconds=chunk_seconds)

    full_text = []
    confs = []
    for i, ch in enumerate(chunks, 1):
        size_mb = os.path.getsize(ch) / (1024 * 1024)
        print(f"[{i}/{len(chunks)}] {Path(ch).name} ({size_mb:.2f} MB)")
        r = transcribe_local_file(ch, language_code=language_code)  # senin mevcut fonksiyon
        if r.transcript:
            full_text.append(r.transcript.strip())
        if r.confidence is not None:
            confs.append(r.confidence)

    return {
        "transcript": "\n".join(full_text),
        "avg_confidence": (sum(confs)/len(confs)) if confs else None,
        "chunks": len(chunks),
    }

def transcribe_local_file(
    input_path: str,
    language_code: str = "en-US",
    enable_punctuation: bool = True,
    use_enhanced: bool = True,
    timeout_sec: int = 300,
) -> TranscriptionResult:
    # Yerel dosyayı (mp3/wav) flac'a çevirip Google Speech-to-Text ile transkribe eder.
    # Not: Çok uzun dosyalarda GCS URI ile async tanıma daha stabil olabilir.
    client = speech.SpeechClient()

    flac_path = ensure_flac_mono_16k(input_path)

    with open(flac_path, "rb") as f:
        content = f.read()

    audio = speech.RecognitionAudio(content=content)

    config = speech.RecognitionConfig(
    encoding=speech.RecognitionConfig.AudioEncoding.FLAC,
    speech_contexts=[speech.SpeechContext(
    phrases=[
        "Prairie Road", "ditch", "wedding", "field",
        "help me", "I don't know where I am",
        "vehicle", "map", "lost"
    ],
    boost=15.0
)]
,
    sample_rate_hertz=16000,
    language_code=language_code,
    enable_automatic_punctuation=True,
    model="phone_call",
    use_enhanced=True,
    audio_channel_count=1,
    max_alternatives=3,
)

    # long_running_recognize: orta/uzun dosyalarda daha güvenli
    operation = client.long_running_recognize(config=config, audio=audio)
    response = client.recognize(config=config, audio=audio)
    parts = []
    confs = []
    for result in response.results:
        if not result.alternatives:
            continue
        alt = result.alternatives[0]
        parts.append(alt.transcript.strip())
        if alt.confidence:
            confs.append(float(alt.confidence))

    transcript = " ".join([p for p in parts if p])
    confidence = (sum(confs) / len(confs)) if confs else None

    return TranscriptionResult(
        transcript=transcript,
        confidence=confidence,
        language_code=language_code,
        flac_path=flac_path,
    )


In [83]:
import re
from pathlib import Path

def call_number(p: Path) -> int:
    m = re.search(r"call_(\d+)$", p.stem)  # call_9 gibi
    return int(m.group(1)) if m else 10**9

mp3_files = sorted(AUDIO_DIR.glob("call_*.mp3"), key=call_number)

# call_9'u seç
sample = next(p for p in mp3_files if call_number(p) == 9)

print("Transcribing (chunked):", sample)
out = transcribe_local_file_chunked(str(sample), language_code="en-US", chunk_seconds=25)

print(out["transcript"][:2000])
print("Chunks:", out["chunks"], "Avg conf:", out["avg_confidence"])


Transcribing (chunked): C:\Users\halil melih\480proje\470_project\bitirme_project\data\raw\911_recordings\audio\call_9.mp3
[1/5] chunk_000.flac (0.74 MB)
[2/5] chunk_001.flac (0.75 MB)
[3/5] chunk_002.flac (0.74 MB)
[4/5] chunk_003.flac (0.74 MB)
[5/5] chunk_004.flac (0.02 MB)
911. What is your emergency? Yes. Um, I need a police officer over here at 7. What's going on? Um, I've got 2 teenage daughters and I just got home from work. They were um physically fighting with each other and 1 of them. Kicked a hole in a door and um, they're 12 and almost 14. And the 12 year old is completely out of control and I I can't I physically
If she's the biggest, I am I can't control her. Okay, did you want us to come over to shoot her? Are you there? Excuse me. Uh, that's a joke. Okay. So how are you? What is your name? Mike forvis. Okay. That's not funny. I can just what I'm going to follow a formal complaint. That's I don't blame you a bit. Hey, did you know what? This is really not very funny. I 